In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import ast, json, time, warnings, os
warnings.filterwarnings('ignore')

start = time.time()

# ============================================================
# 1. ЗАГРУЗКА И ОЧИСТКА ДАННЫХ (теперь через kagglehub)
# ============================================================
try:
    import kagglehub
except ImportError:
    !pip install kagglehub -q
    import kagglehub

# Скачиваем датасет (если уже загружен в кеш, использует кеш)
print("Загрузка датасета через kagglehub...")
dataset_path = kagglehub.dataset_download("rounakbanik/the-movies-dataset")
print(f"Данные сохранены в: {dataset_path}")

ratings_path = os.path.join(dataset_path, "ratings.csv")
links_path   = os.path.join(dataset_path, "links.csv")
movies_path  = os.path.join(dataset_path, "movies_metadata.csv")

ratings = pd.read_csv(ratings_path)
links   = pd.read_csv(links_path)
movies  = pd.read_csv(movies_path)

# --- Парсинг JSON-подобных колонок (ast.literal_eval) ---
def parse_json_column(val):
    if isinstance(val, (list, dict)):
        return val
    if pd.isna(val) or val == '' or val == '[]' or val == '{}':
        return []
    try:
        data = ast.literal_eval(str(val))
        if isinstance(data, list):
            return [item['name'] for item in data if isinstance(item, dict) and 'name' in item]
        if isinstance(data, dict):
            return data.get('name', '')
        return []
    except (ValueError, SyntaxError):
        return []

movies['genres'] = movies['genres'].apply(parse_json_column)

# --- Очистка movies ---
movies['id_clean'] = pd.to_numeric(movies['id'], errors='coerce')
invalid_mask = (
    movies['id_clean'].isna() |
    movies['vote_average'].isna() |
    movies['vote_count'].isna() |
    movies.duplicated(subset=['id_clean'], keep='first')
)
movies = movies[~invalid_mask].copy()
movies['id'] = movies['id_clean'].astype(int)
movies = movies.drop(columns=['id_clean'])
movies = movies[['id', 'title', 'genres']]
print(f"Загружено фильмов: {len(movies):,}")

# --- Очистка links ---
links = links.dropna(subset=['tmdbId']).drop_duplicates(subset=['tmdbId'], keep='first')
valid_movie_ids = set(movies['id'])
links = links[links['tmdbId'].astype(float).astype(int).isin(valid_movie_ids)]
links = links.rename(columns={'tmdbId': 'id'})
links['id'] = links['id'].astype(int)

# --- Присоединение tmdbId к рейтингам ---
ratings = ratings.dropna(subset=['userId', 'movieId', 'rating'])
ratings = ratings.merge(links[['movieId', 'id']], on='movieId', how='inner')
ratings = ratings.rename(columns={'id': 'tmdbId'})
ratings = ratings.drop(columns=['movieId'])
ratings = ratings[ratings['tmdbId'].isin(valid_movie_ids)]
ratings['tmdbId'] = ratings['tmdbId'].astype(int)

# --- Фильтрация по минимальному числу оценок ---
movie_counts = ratings['tmdbId'].value_counts()
ratings = ratings[ratings['tmdbId'].isin(movie_counts[movie_counts >= 3].index)]
user_counts = ratings['userId'].value_counts()
ratings = ratings[ratings['userId'].isin(user_counts[user_counts >= 5].index)]
print(f"🔗 После фильтрации: {ratings.shape[0]:,} оценок, "
      f"пользователей: {ratings['userId'].nunique():,}, фильмов: {ratings['tmdbId'].nunique():,}")

movie_genres = dict(zip(movies['id'], movies['genres']))

# ============================================================
# 2. ИНДЕКСЫ И РАЗБИЕНИЕ
# ============================================================
users = ratings['userId'].unique()
items = ratings['tmdbId'].unique()
user2idx = {u: i for i, u in enumerate(users)}
item2idx = {m: i for i, m in enumerate(items)}
idx2item = {i: m for m, i in item2idx.items()}

ratings['user_idx'] = ratings['userId'].map(user2idx)
ratings['item_idx'] = ratings['tmdbId'].map(item2idx)

n_users = len(users)
n_items = len(items)

train, test = train_test_split(ratings, test_size=0.2, random_state=42)
print(f"Train: {len(train):,} оценок, Test: {len(test):,} оценок")
print(f"Подготовка данных: {time.time()-start:.1f} сек.\n")

THRESHOLD = 3.5
test_pos_per_user = test[test['rating'] >= THRESHOLD].groupby('userId').size()
print(f"Тестовых положительных (>=3.5) на пользователя: "
      f"среднее = {test_pos_per_user.mean():.2f}, медиана = {test_pos_per_user.median():.1f}, "
      f"макс = {test_pos_per_user.max()}, доля с >=1 = { (test_pos_per_user>0).mean():.2%}")

# ==================================
# 3. IMPLICIT ALS
# ==================================
K = 50
alpha = 150
lambda_reg = 0.3
n_iter = 15

print(f"📌 ALS: K={K}, alpha={alpha}, λ={lambda_reg}, итераций={n_iter}")

train_pos = train[train['rating'] >= THRESHOLD].copy()
train_dict = {}
for _, row in train_pos.iterrows():
    u = int(row['user_idx'])
    i = int(row['item_idx'])
    train_dict.setdefault(u, []).append(i)

np.random.seed(42)
U = np.random.normal(0, 0.1, (n_users, K))
V = np.random.normal(0, 0.1, (n_items, K))
I = np.eye(K)

for it in range(n_iter):
    start_it = time.time()
    
    global_VtV = V.T @ V
    for u, items_u in train_dict.items():
        Vi = V[items_u]
        A = global_VtV + alpha * (Vi.T @ Vi) + lambda_reg * I
        b = (1 + alpha) * np.sum(Vi, axis=0)
        try:
            U[u] = np.linalg.solve(A, b)
        except:
            U[u] = np.linalg.lstsq(A, b, rcond=None)[0]
    
    item_dict = {}
    for u, items_u in train_dict.items():
        for i in items_u:
            item_dict.setdefault(i, []).append(u)
    
    global_UtU = U.T @ U
    for i, users_i in item_dict.items():
        Uj = U[users_i]
        A = global_UtU + alpha * (Uj.T @ Uj) + lambda_reg * I
        b = (1 + alpha) * np.sum(Uj, axis=0)
        try:
            V[i] = np.linalg.solve(A, b)
        except:
            V[i] = np.linalg.lstsq(A, b, rcond=None)[0]
    
    print(f"  Итерация {it+1:2d} за {time.time()-start_it:.1f} сек")

print(f"Обучение ALS: {time.time()-start:.1f} сек\n")



📥 Загрузка датасета через kagglehub...
📂 Данные сохранены в: /kaggle/input/datasets/rounakbanik/the-movies-dataset
✅ Загружено фильмов: 45,430
🔗 После фильтрации: 25,929,633 оценок, пользователей: 256,044, фильмов: 32,237
📊 Train: 20,743,706 оценок, Test: 5,185,927 оценок
⏱️ Подготовка данных: 35.5 сек.

📊 Тестовых положительных (>=3.5) на пользователя: среднее = 13.93, медиана = 5.0, макс = 1560, доля с >=1 = 100.00%
📌 ALS: K=50, alpha=150, λ=0.3, итераций=15
  Итерация  1 за 48.2 сек
  Итерация  2 за 49.2 сек
  Итерация  3 за 49.1 сек
  Итерация  4 за 49.1 сек
  Итерация  5 за 49.0 сек
  Итерация  6 за 50.1 сек
  Итерация  7 за 48.7 сек
  Итерация  8 за 48.1 сек
  Итерация  9 за 47.0 сек
  Итерация 10 за 47.4 сек
  Итерация 11 за 47.7 сек
  Итерация 12 за 47.1 сек
  Итерация 13 за 46.8 сек
  Итерация 14 за 47.3 сек
  Итерация 15 за 47.3 сек
⏱️ Обучение ALS: 1130.7 сек



In [2]:
# ==================================
# 4. ГИБРИД ALS + ЖАНРЫ + SERENDIPITY
# ==================================
N_CANDIDATES = 2000
W_ALS = 0.8
print(f"📊 N_CANDIDATES={N_CANDIDATES}, W_ALS={W_ALS}")

user_rated_items = {u: set(items) for u, items in train_dict.items()}
for u in range(n_users):
    if u not in user_rated_items:
        user_rated_items[u] = set()

all_items_idx = np.arange(n_items)

print("   Построение жанровых профилей...")
user_genre_prefs = {}
for u_idx in range(n_users):
    liked_items = train_dict.get(u_idx, [])
    genre_counter = {}
    for i in liked_items:
        tmdb_id = idx2item[i]
        for g in movie_genres.get(tmdb_id, []):
            genre_counter[g] = genre_counter.get(g, 0) + 1
    total = sum(genre_counter.values())
    if total > 0:
        user_genre_prefs[u_idx] = {g: cnt/total for g, cnt in genre_counter.items()}
    else:
        user_genre_prefs[u_idx] = {}
print("   Профили готовы.")

pop_counts = train_pos['item_idx'].value_counts()
pop_sorted_items = pop_counts.index.values

test_grouped = test[test['rating'] >= THRESHOLD].groupby('userId')['tmdbId'].apply(list).to_dict()
test_users = test['userId'].unique()

def hybrid_score(als_norm, tmdb_id, u_idx):
    genres = movie_genres.get(tmdb_id, [])
    if not genres:
        return als_norm
    prefs = user_genre_prefs.get(u_idx, {})
    genre_sim = sum(prefs.get(g, 0.0) for g in genres)
    return W_ALS * als_norm + (1 - W_ALS) * genre_sim

def calc_metrics_for_user(rec_tmdb, actual_tmdb, k_vals):
    p, r, n = {}, {}, {}
    for k in k_vals:
        rec_set = set(rec_tmdb[:k])
        act_set = set(actual_tmdb)
        p[k] = len(rec_set & act_set) / k
        r[k] = len(rec_set & act_set) / len(act_set) if act_set else 0.0
        dcg = sum(1.0 / np.log2(rank+2) for rank, item in enumerate(rec_tmdb[:k]) if item in act_set)
        idcg = sum(1.0 / np.log2(rank+2) for rank in range(min(k, len(act_set))))
        n[k] = dcg / idcg if idcg > 0 else 0.0
    return p, r, n

# ---------- Функции для serendipity ----------
def surprise_item(tmdb_id, u_idx, user_genre_prefs, movie_genres):
    """
    Неожиданность фильма tmdb_id для пользователя u_idx (0..1).
    Чем меньше пересечение с предпочтениями, тем выше surprise.
    """
    prefs = user_genre_prefs.get(u_idx, {})
    if not prefs:
        return 1.0  # нет истории → всё неожиданно
    total_pref = sum(prefs.values())
    if total_pref == 0:
        return 1.0

    genres = movie_genres.get(tmdb_id, [])
    if not genres:
        return 1.0  # нет жанров → считаем неожиданным

    match = sum(prefs.get(g, 0.0) for g in genres)
    surprise = 1.0 - (match / total_pref)
    return max(0.0, min(1.0, surprise))

def serendipity_at_k(rec_tmdb, actual_tmdb, u_idx, user_genre_prefs, movie_genres, k):
    """
    Serendipity@k: средняя неожиданность среди первых k рекомендаций,
    которые оказались релевантными. Если релевантных нет -> 0.
    """
    if not actual_tmdb:
        return 0.0
    rec_k = rec_tmdb[:k]
    relevant_set = set(actual_tmdb)
    ser_sum = 0.0
    n_rel = 0
    for item in rec_k:
        if item in relevant_set:
            ser_sum += surprise_item(item, u_idx, user_genre_prefs, movie_genres)
            n_rel += 1
    if n_rel == 0:
        return 0.0
    return ser_sum / n_rel   # можно заменить на ser_sum / k, если нужно наказывать за нерелевантные

# ---------------------------------------------

k_values = [5, 10, 15]
prec_hybrid = {k: [] for k in k_values}
rec_hybrid  = {k: [] for k in k_values}
ndcg_hybrid = {k: [] for k in k_values}
prec_pop = {k: [] for k in k_values}
rec_pop  = {k: [] for k in k_values}
ndcg_pop = {k: [] for k in k_values}
ser_hybrid = {k: [] for k in k_values}   # ← новая метрика
ser_pop    = {k: [] for k in k_values}   # ← новая метрика

eval_users = 0

print(f"\n   Оценка для {len(test_users)} тестовых пользователей...")
start_eval = time.time()

for idx, user_id in enumerate(test_users, 1):
    u_idx = user2idx[user_id]
    actual_tmdb = test_grouped.get(user_id, [])
    if not actual_tmdb:
        continue

    exclude = user_rated_items.get(u_idx, set())
    candidate_mask = np.ones(n_items, dtype=bool)
    candidate_mask[list(exclude)] = False
    candidates = all_items_idx[candidate_mask]
    if len(candidates) == 0:
        continue

    # ALS scores
    als_scores_raw = U[u_idx] @ V[candidates].T
    min_als = np.min(als_scores_raw)
    max_als = np.max(als_scores_raw)
    if max_als > min_als:
        als_scores = (als_scores_raw - min_als) / (max_als - min_als)
    else:
        als_scores = np.zeros_like(als_scores_raw)

    if len(candidates) > N_CANDIDATES:
        top_indices = np.argpartition(als_scores, -N_CANDIDATES)[-N_CANDIDATES:]
        top_indices = top_indices[np.argsort(als_scores[top_indices])[::-1]]
    else:
        top_indices = np.argsort(als_scores)[::-1]

    top_candidates = candidates[top_indices]
    top_scores_norm = als_scores[top_indices]

    # Гибридный скоринг
    hybrid_scores = np.array([
        hybrid_score(top_scores_norm[i], idx2item[top_candidates[i]], u_idx)
        for i in range(len(top_candidates))
    ])
    final_order = np.argsort(hybrid_scores)[::-1]
    final_candidates = top_candidates[final_order[:max(k_values)]]
    rec_hybrid_tmdb = [idx2item[i] for i in final_candidates]

    # Top‑Pop baseline
    pop_filtered = [i for i in pop_sorted_items if i not in exclude]
    top_pop_items = pop_filtered[:max(k_values)]
    rec_pop_tmdb = [idx2item[i] for i in top_pop_items]

    # Базовые метрики
    p_h, r_h, n_h = calc_metrics_for_user(rec_hybrid_tmdb, actual_tmdb, k_values)
    p_pop, r_pop, n_pop = calc_metrics_for_user(rec_pop_tmdb, actual_tmdb, k_values)

    # Serendipity для гибрида
    for k in k_values:
        ser_hybrid[k].append(
            serendipity_at_k(rec_hybrid_tmdb, actual_tmdb, u_idx,
                             user_genre_prefs, movie_genres, k)
        )
    # Serendipity для Top‑Pop
    for k in k_values:
        ser_pop[k].append(
            serendipity_at_k(rec_pop_tmdb, actual_tmdb, u_idx,
                             user_genre_prefs, movie_genres, k)
        )

    # Накопление precision/recall/ndcg
    eval_users += 1
    for k in k_values:
        prec_hybrid[k].append(p_h[k])
        rec_hybrid[k].append(r_h[k])
        ndcg_hybrid[k].append(n_h[k])
        prec_pop[k].append(p_pop[k])
        rec_pop[k].append(r_pop[k])
        ndcg_pop[k].append(n_pop[k])

    if idx % 500 == 0 or idx == len(test_users):
        elapsed = time.time() - start_eval
        speed = idx / elapsed
        rem = (len(test_users) - idx) / speed if speed > 0 else 0
        print(f"   Прогресс: {idx}/{len(test_users)} ({100*idx/len(test_users):.1f}%) | "
              f"оценено {eval_users} | осталось ~{rem/60:.1f} мин")

print(f"\n⏱️ Оценка завершена за {(time.time()-start_eval)/60:.1f} мин. Оценено {eval_users} пользователей.\n")

# === Вывод результатов ===
print("=== Мягкий гибрид (ALS + жанры) ===")
for k in k_values:
    print(f"k={k:2d} | Precision: {np.mean(prec_hybrid[k]):.4f} | Recall: {np.mean(rec_hybrid[k]):.4f} | NDCG: {np.mean(ndcg_hybrid[k]):.4f}")

print("\n=== Top‑Pop baseline ===")
for k in k_values:
    print(f"k={k:2d} | Precision: {np.mean(prec_pop[k]):.4f} | Recall: {np.mean(rec_pop[k]):.4f} | NDCG: {np.mean(ndcg_pop[k]):.4f}")

print("\n--- Отношение Гибрид / Top‑Pop ---")
for k in k_values:
    p_ratio = np.mean(prec_hybrid[k]) / np.mean(prec_pop[k]) if np.mean(prec_pop[k]) > 0 else float('inf')
    r_ratio = np.mean(rec_hybrid[k]) / np.mean(rec_pop[k]) if np.mean(rec_pop[k]) > 0 else float('inf')
    print(f"k={k:2d} | Precision: {p_ratio:.2f}x | Recall: {r_ratio:.2f}x")

print("\n=== Serendipity (средняя неожиданность среди релевантных) ===")
print("Гибрид:")
for k in k_values:
    print(f"  k={k:2d} | {np.mean(ser_hybrid[k]):.4f}")
print("Top‑Pop:")
for k in k_values:
    print(f"  k={k:2d} | {np.mean(ser_pop[k]):.4f}")

📊 N_CANDIDATES=2000, W_ALS=0.8
   Построение жанровых профилей...
   Профили готовы.

   Оценка для 247627 тестовых пользователей...
   Прогресс: 500/247627 (0.2%) | оценено 494 | осталось ~85.0 мин
   Прогресс: 1000/247627 (0.4%) | оценено 993 | осталось ~84.7 мин
   Прогресс: 1500/247627 (0.6%) | оценено 1486 | осталось ~84.3 мин
   Прогресс: 2000/247627 (0.8%) | оценено 1983 | осталось ~84.2 мин
   Прогресс: 2500/247627 (1.0%) | оценено 2479 | осталось ~83.9 мин
   Прогресс: 3000/247627 (1.2%) | оценено 2976 | осталось ~83.7 мин
   Прогресс: 3500/247627 (1.4%) | оценено 3468 | осталось ~83.5 мин
   Прогресс: 4000/247627 (1.6%) | оценено 3965 | осталось ~83.5 мин
   Прогресс: 4500/247627 (1.8%) | оценено 4460 | осталось ~83.3 мин
   Прогресс: 5000/247627 (2.0%) | оценено 4957 | осталось ~83.1 мин
   Прогресс: 5500/247627 (2.2%) | оценено 5454 | осталось ~83.0 мин
   Прогресс: 6000/247627 (2.4%) | оценено 5950 | осталось ~82.8 мин
   Прогресс: 6500/247627 (2.6%) | оценено 6447 | остал